# SOUL AI — General Training Lab (90 minutes)

**Target repository:** `GhastlyVN/Soul-AI`  
**Audited source revision:** `92a3a1f9d3dde0cd6e6c60122bb0b783d289cb51`  
**Runtime:** Google Colab, automatic GPU/CPU selection  
**Mode:** Full training + independent final evaluation

This notebook preserves the repository's existing architecture. It trains the existing RSSM world model with `scripts.phase0_train`, populates the existing `MemoryManager` from real collected transitions, and runs the existing MiniCraft compressed/mapped skill pipelines.

## Important research boundaries

- TRAIN seeds: `0..999`; this run uses a fresh subrange.
- DEV seeds: `1000..1999`; validation and the final independent DEV challenge use separate subranges.
- CONFIRM is **not defined** in the repository protocol and is never accessed.
- MiniCraft skills remain Candidate/exploratory; no promotion or compute-saving claim is made.
- The repository does not contain a language-model/text-reasoning training stack. Text reasoning, pixel perception, and general intelligence are not claimed or scored.
- The final evaluator is separate from the trainer. It checks a frozen checkpoint and records negative results without tuning on the final exam.

The 90-minute setting is a **maximum session budget**, not a promise that Colab will allocate a particular GPU or that training will consume the entire window. Colab resource availability and runtime limits vary.

In [ ]:
# 0) Run configuration — edit only if you intentionally want a different experiment.
from pathlib import Path
import os, sys, json, time, shutil, hashlib, subprocess, platform
from datetime import datetime, timezone

REPO_URL = "https://github.com/GhastlyVN/Soul-AI.git"
PINNED_COMMIT = "92a3a1f9d3dde0cd6e6c60122bb0b783d289cb51"

TIME_BUDGET_SECONDS = 90 * 60
TRAIN_BUDGET_CAP_SECONDS = 45 * 60
MAX_TRAIN_UPDATES = 12_000
CALIBRATION_UPDATES = 12

# Disjoint, protocol-compliant seeds. CONFIRM is intentionally absent.
WM_TRAIN_SEEDS = list(range(600, 680))       # 80 TRAIN episodes
DEV_VALIDATION_SEEDS = list(range(1500, 1508))
DEV_FINAL_EXAM_SEEDS = list(range(1800, 1816))

SKILL_TRAIN_SEEDS = list(range(700, 720))
SKILL_EVAL_SEEDS = list(range(730, 740))
MAPPED_SKILL_EVAL_SEEDS = list(range(750, 756))

STEPS_PER_TRAIN_SEED = 240
MAX_TICKS = 300
WM_MAX_TICKS = 100
HIDDEN_DIM = 64
LATENT_DIM = 16
SEQ_LEN = 16
BATCH_SIZE = 16

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
LOCAL_ROOT = Path("/content/soul-general-training")
RUN_DIR = LOCAL_ROOT / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)
RUN_STARTED = time.monotonic()
print("Run:", RUN_ID)
print("Pinned source:", PINNED_COMMIT)
print("Time budget:", TIME_BUDGET_SECONDS // 60, "minutes")

In [ ]:
# 1) Clone the exact audited revision (not whichever main happens to exist later).
REPO_DIR = Path("/content/Soul-AI")
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(["git", "clone", "--filter=blob:none", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "fetch", "--depth=1", "origin", PINNED_COMMIT], cwd=REPO_DIR, check=True)
subprocess.run(["git", "checkout", "--detach", PINNED_COMMIT], cwd=REPO_DIR, check=True)
os.chdir(REPO_DIR)

ACTUAL_COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
assert ACTUAL_COMMIT == PINNED_COMMIT, (ACTUAL_COMMIT, PINNED_COMMIT)
print("Checked out:", ACTUAL_COMMIT)
print("Working tree:", subprocess.check_output(["git", "status", "--short"], text=True).strip() or "clean")

In [ ]:
# 2) Detect hardware; keep Colab's preinstalled CUDA-enabled PyTorch.
import importlib.util
missing = [name for name, module in [("pytest", "pytest"), ("yaml", "yaml")] if importlib.util.find_spec(module) is None]
if missing:
    packages = ["pytest" if x == "pytest" else "PyYAML" for x in missing]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)

import numpy as np
import torch
torch.set_num_threads(max(1, min(4, os.cpu_count() or 2)))

if torch.cuda.is_available():
    DEVICE = "cuda"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

RUNTIME = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "numpy": np.__version__,
    "device": DEVICE,
    "cuda_available": bool(torch.cuda.is_available()),
    "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "cpu_count": os.cpu_count(),
    "threads": torch.get_num_threads(),
}
print(json.dumps(RUNTIME, indent=2))

In [ ]:
# 3) Mount Drive for durable artifacts and audit the actual repository modules.
from google.colab import drive
drive.mount("/content/drive")

DRIVE_RUN_DIR = Path("/content/drive/MyDrive/SOUL_General_Training") / RUN_ID
DRIVE_RUN_DIR.mkdir(parents=True, exist_ok=False)

def persist_file(path):
    path = Path(path)
    if path.exists() and path.is_file():
        target = DRIVE_RUN_DIR / path.relative_to(RUN_DIR)
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(path, target)
        return target
    return None

required_paths = [
    "soul/model.py", "soul/obs.py", "soul/soul.py", "soul/trainer.py",
    "soul/memory.py", "soul/plan.py", "scripts/phase0_collect.py",
    "scripts/phase0_train.py", "scripts/phase0_score.py",
    "scripts/phase0_validate_protocol.py", "minicraft/env.py",
    "minicraft/safety.py", "minicraft/compressed_skill.py",
    "scripts/phase1_compressed_skill.py", "scripts/phase1_mapped_skill.py",
    "configs/seeds.yaml", "configs/phase0_protocol_draft.yaml",
]
missing_paths = [p for p in required_paths if not (REPO_DIR / p).is_file()]
assert not missing_paths, f"Audited architecture is incomplete: {missing_paths}"

from soul.soul import Soul, SoulBrain
from soul.memory import MemoryManager
from soul.config import Config
from minicraft import MiniCraftEnv, MiniCraftSafetyController
from minicraft.env import DIRECTIONS, LAVA, PATCH_RADIUS
from scripts.phase0_collect import collect
from scripts.phase0_train import train
from scripts.phase0_score import score

print("Architecture audit: PASS")
print("Detected: RSSM model, ObsEncoder, Trainer, Soul/SoulBrain, ReplayBuffer, MemoryManager, MiniCraft, safety, compressed/mapped skills.")
print("No language-model module or pixel encoder is present in this revision.")

In [ ]:
# 4) Protocol validation + full repository regression suite. Fail closed before training.
def run_checked(command, timeout):
    print("$", " ".join(map(str, command)))
    result = subprocess.run(command, cwd=REPO_DIR, text=True, capture_output=True, timeout=timeout)
    print(result.stdout[-12000:])
    if result.stderr:
        print(result.stderr[-6000:])
    if result.returncode != 0:
        raise RuntimeError(f"Command failed ({result.returncode}): {command}")
    return result

run_checked([sys.executable, "-m", "scripts.phase0_validate_protocol"], timeout=180)
test_start = time.monotonic()
test_result = run_checked([sys.executable, "-m", "pytest", "-q"], timeout=900)
TEST_SECONDS = time.monotonic() - test_start
print(f"Regression suite passed in {TEST_SECONDS:.1f}s")

In [ ]:
# 5) Collect real MiniCraft TRAIN transitions using the repository's safety-filtered collector.
DATA_PATH = RUN_DIR / "train_minicraft.jsonl"
transition_count = collect(
    DATA_PATH,
    WM_TRAIN_SEEDS,
    steps_per_seed=STEPS_PER_TRAIN_SEED,
    max_ticks=MAX_TICKS,
)
assert transition_count > 0
data_sha256 = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
print("Transitions:", transition_count)
print("Episodes/seeds:", len(WM_TRAIN_SEEDS))
print("Dataset SHA-256:", data_sha256)
persist_file(DATA_PATH)

# Ensure collection stayed inside the locked TRAIN range.
with DATA_PATH.open("r", encoding="utf-8") as f:
    first_rows = [json.loads(next(f)) for _ in range(3)]
assert all(r["split"] == "train" and 0 <= int(r["seed"]) <= 999 for r in first_rows)
print("TRAIN-only data boundary: PASS")

In [ ]:
# 6) Calibrate update speed, then train the existing RSSM architecture within the 90-minute budget.
CALIBRATION_CKPT = RUN_DIR / "calibration_only.pt"
cal_start = time.monotonic()
cal_report = train(
    DATA_PATH, CALIBRATION_CKPT,
    updates=CALIBRATION_UPDATES,
    hidden_dim=HIDDEN_DIM, latent_dim=LATENT_DIM,
    seq_len=SEQ_LEN, batch_size=BATCH_SIZE,
    seed=20261125, device=DEVICE,
)
cal_seconds = time.monotonic() - cal_start
seconds_per_update = max(cal_seconds / CALIBRATION_UPDATES, 1e-4)

elapsed = time.monotonic() - RUN_STARTED
remaining = max(0, TIME_BUDGET_SECONDS - elapsed)
# Reserve at least 25 minutes for memory, skill pipelines, final exam and export.
if remaining < 30 * 60:
    raise RuntimeError("Less than 30 minutes remain; stopping before a partial training run.")
train_budget = min(TRAIN_BUDGET_CAP_SECONDS, remaining - 25 * 60)
train_budget = max(5 * 60, train_budget)
estimated_updates = int((train_budget / seconds_per_update) * 0.55)
TRAIN_UPDATES = max(500, min(MAX_TRAIN_UPDATES, estimated_updates))
print({
    "calibration_seconds": round(cal_seconds, 2),
    "seconds_per_update": round(seconds_per_update, 4),
    "training_budget_seconds": int(train_budget),
    "selected_updates": TRAIN_UPDATES,
    "device": DEVICE,
})

CHECKPOINT_PATH = RUN_DIR / "soul_rssm_trained.pt"
TRAIN_REPORT_PATH = RUN_DIR / "train_report.json"
train_start = time.monotonic()
TRAIN_REPORT = train(
    DATA_PATH, CHECKPOINT_PATH,
    updates=TRAIN_UPDATES,
    hidden_dim=HIDDEN_DIM, latent_dim=LATENT_DIM,
    seq_len=SEQ_LEN, batch_size=BATCH_SIZE,
    seed=20261126, device=DEVICE,
)
TRAIN_SECONDS = time.monotonic() - train_start
TRAIN_REPORT["wall_seconds"] = TRAIN_SECONDS
TRAIN_REPORT["runtime"] = RUNTIME
TRAIN_REPORT_PATH.write_text(json.dumps(TRAIN_REPORT, indent=2, ensure_ascii=False, allow_nan=False), encoding="utf-8")
persist_file(CHECKPOINT_PATH)
persist_file(TRAIN_REPORT_PATH)
print("Training complete:", TRAIN_REPORT["updates"], "updates in", round(TRAIN_SECONDS, 1), "seconds")
print("Checkpoint SHA-256:", TRAIN_REPORT["checkpoint_sha256"])
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
# 7) Exploratory validation on a separate DEV slice. This is not the final exam or CONFIRM.
VALIDATION_REPORT = score(
    CHECKPOINT_PATH,
    RUN_DIR / "validation_dev.json",
    DEV_VALIDATION_SEEDS,
    episodes_per_seed=1,
    max_ticks=WM_MAX_TICKS,
    shuffle_repeats=5,
    raw_output=RUN_DIR / "validation_predictions.jsonl",
    expected_checkpoint_sha256=TRAIN_REPORT["checkpoint_sha256"],
)
persist_file(RUN_DIR / "validation_dev.json")
persist_file(RUN_DIR / "validation_predictions.jsonl")

validation_h = VALIDATION_REPORT["equal_episode_mean"]["horizons"]
print("Validation (exploratory DEV only):")
for horizon in ("1", "5", "20", "100"):
    row = validation_h[horizon]
    print(horizon, {
        "anchored_model_mse": row["anchored_model_mse"],
        "open_loop_model_mse": row["open_loop_model_mse"],
        "persistence_mse": row["persistence_mse"],
    })
print("Uncertainty:", VALIDATION_REPORT.get("uncertainty", "NOT_REPORTED"))
print("No threshold selection or protocol lock is performed.")

In [ ]:
# 8) Populate the existing long-term memory from collected transitions; do not invent a new memory system.
# Load the exact trained model/config, then feed actual observations/actions/rewards to MemoryManager.
checkpoint_payload = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
soul_cfg = Config.from_dict(checkpoint_payload["config"])
soul_cfg.device = DEVICE
trained_soul = Soul(soul_cfg, checkpoint=str(CHECKPOINT_PATH))
trained_soul.enc.freeze_schema()

memory_start = time.monotonic()
memory_events = 0
with DATA_PATH.open("r", encoding="utf-8") as stream:
    for line in stream:
        row = json.loads(line)
        event = {
            "time": memory_events,
            "action": np.eye(9, dtype=np.float32)[int(row["action"])].tolist(),
            "reward": float(row["reward"]),
            "done": bool(row["done"]),
            "goal": None,
            "context": {
                "goal": "minicraft_train_transition",
                "seed": int(row["seed"]),
                "tick": int(row["tick"]),
                "observation": row["observation"],
            },
            "tags": ["colab_training", "minicraft"],
        }
        important = bool(event["done"] or abs(event["reward"]) > 0.05)
        trained_soul.memory_manager.record_event(event, important=important)
        if important:
            trained_soul.memory_manager.extract_skill_from_event(event, threshold=0.05)
        memory_events += 1

MEMORY_PATH = RUN_DIR / "memory_manager.json"
trained_soul.memory_manager.save(str(MEMORY_PATH))
memory_seconds = time.monotonic() - memory_start

# Independent round-trip check through a fresh MemoryManager instance.
memory_reloaded = MemoryManager()
memory_reloaded.load(str(MEMORY_PATH))
memory_roundtrip = {
    "events_before": len(trained_soul.memory_manager.archive.events),
    "events_after": len(memory_reloaded.archive.events),
    "skills_before": len(trained_soul.memory_manager.skills.skills),
    "skills_after": len(memory_reloaded.skills.skills),
    "roundtrip_ok": (
        len(trained_soul.memory_manager.archive.events) == len(memory_reloaded.archive.events)
        and len(trained_soul.memory_manager.skills.skills) == len(memory_reloaded.skills.skills)
    ),
    "source_transitions_seen": memory_events,
    "wall_seconds": memory_seconds,
}
assert memory_roundtrip["roundtrip_ok"]
persist_file(MEMORY_PATH)
print(json.dumps(memory_roundtrip, indent=2))

In [ ]:
# 9) Train/evaluate the existing compressed skill and evaluate the observation-mapped skill.
# Both pipelines use TRAIN seeds only; evaluation seeds are disjoint from their training seeds.
from scripts.phase1_compressed_skill import run as run_compressed_skill
from scripts.phase1_mapped_skill import run as run_mapped_skill

SKILL_DIR = RUN_DIR / "compressed_skill"
skill_start = time.monotonic()
COMPRESSED_REPORT = run_compressed_skill(
    SKILL_DIR,
    train_seeds=SKILL_TRAIN_SEEDS,
    eval_seeds=SKILL_EVAL_SEEDS,
    max_ticks=MAX_TICKS,
)
COMPRESSED_SECONDS = time.monotonic() - skill_start

MAPPED_DIR = RUN_DIR / "mapped_skill"
mapped_start = time.monotonic()
MAPPED_REPORT = run_mapped_skill(
    MAPPED_SKILL_EVAL_SEEDS,
    MAPPED_DIR,
    max_ticks=MAX_TICKS,
)
MAPPED_SECONDS = time.monotonic() - mapped_start

print("Compressed candidate:")
for task, result in COMPRESSED_REPORT["tasks"].items():
    print(task, "success=", result["success_fraction"],
          "learned_agreement=", result["learned_agreement_fraction"])
print("Mapped candidate:")
for task, result in MAPPED_REPORT["tasks"].items():
    print(task, result)
print("Skill pipeline seconds:", round(COMPRESSED_SECONDS + MAPPED_SECONDS, 1))
print("Interpretation: Candidate/exploratory only; compressed tree is advisory and gated by mapped-planner agreement.")

In [ ]:
# 10) Freeze model artifact and record provenance BEFORE the independent final exam.
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

FROZEN_CHECKPOINT_SHA256 = sha256_file(CHECKPOINT_PATH)
assert FROZEN_CHECKPOINT_SHA256 == TRAIN_REPORT["checkpoint_sha256"]
FROZEN_MANIFEST = {
    "schema_version": 1,
    "status": "FROZEN_BEFORE_FINAL_EXAM",
    "run_id": RUN_ID,
    "repo": REPO_URL,
    "commit": ACTUAL_COMMIT,
    "runtime": RUNTIME,
    "train_seeds": WM_TRAIN_SEEDS,
    "dev_validation_seeds": DEV_VALIDATION_SEEDS,
    "final_exam_dev_seeds": DEV_FINAL_EXAM_SEEDS,
    "confirm_accessed": False,
    "updates": TRAIN_REPORT["updates"],
    "dataset_sha256": data_sha256,
    "checkpoint_sha256": FROZEN_CHECKPOINT_SHA256,
    "memory_archive_sha256": sha256_file(MEMORY_PATH),
    "compressed_skill_manifest_sha256": sha256_file(SKILL_DIR / "manifest.json"),
    "mapped_skill_manifest_sha256": sha256_file(MAPPED_DIR / "manifest.json"),
    "limits": [
        "DEV validation and final exam are exploratory, not CONFIRM.",
        "No formal Phase 0 protocol lock or Phase 5 adaptive-compute claim.",
        "MiniCraft Candidate skills are not promoted.",
    ],
}
FREEZE_PATH = RUN_DIR / "frozen_manifest.json"
FREEZE_PATH.write_text(json.dumps(FROZEN_MANIFEST, indent=2, ensure_ascii=False), encoding="utf-8")
persist_file(FREEZE_PATH)
persist_file(MEMORY_PATH)
shutil.copytree(SKILL_DIR, DRIVE_RUN_DIR / "compressed_skill", dirs_exist_ok=True)
shutil.copytree(MAPPED_DIR, DRIVE_RUN_DIR / "mapped_skill", dirs_exist_ok=True)
print("Frozen checkpoint:", FROZEN_CHECKPOINT_SHA256)

In [ ]:
# 11) Independent final exam — run only after freeze; do not tune on these results.
exam_start = time.monotonic()
FINAL_WM_REPORT = score(
    CHECKPOINT_PATH,
    RUN_DIR / "final_exam_world_model.json",
    DEV_FINAL_EXAM_SEEDS,
    episodes_per_seed=1,
    max_ticks=WM_MAX_TICKS,
    shuffle_repeats=5,
    raw_output=RUN_DIR / "final_exam_predictions.jsonl",
    expected_checkpoint_sha256=FROZEN_CHECKPOINT_SHA256,
)

# Test the actual SoulBrain perception/prediction/planning interfaces on a fresh MiniCraft state.
exam_env = MiniCraftEnv(seed=DEV_FINAL_EXAM_SEEDS[0], max_ticks=WM_MAX_TICKS)
exam_obs = exam_env.observe()
perceived = trained_soul.brain.perceive(exam_obs)
planned = np.asarray(trained_soul.brain.plan(exam_obs, n_samples=16, deterministic=True), dtype=np.float32).reshape(-1)
assert perceived.ndim == 1 and np.isfinite(perceived).all()
assert planned.size == 9 and np.isfinite(planned).all()
planned_action = int(np.argmax(planned))
predicted_effect = np.asarray(
    trained_soul.brain.predict(exam_obs, np.eye(9, dtype=np.float32)[planned_action]),
    dtype=np.float32,
)
brain_interface_exam = {
    "perception_finite": bool(np.isfinite(perceived).all()),
    "planning_action": planned_action,
    "planning_output_shape": list(planned.shape),
    "prediction_finite": bool(np.isfinite(predicted_effect).all()),
    "prediction_shape": list(predicted_effect.shape),
}
assert brain_interface_exam["prediction_finite"]

# Safety contract: every movement direction is blocked when the adjacent target is lava.
safety = MiniCraftSafetyController()
safety_checks = []
for action, (dx, dy) in DIRECTIONS.items():
    altered = dict(exam_obs)
    patch = np.array(exam_obs["patch"], copy=True)
    patch[PATCH_RADIUS + dy, PATCH_RADIUS + dx] = LAVA
    altered["patch"] = patch
    decision = safety.filter(altered, int(action))
    safety_checks.append({
        "action": int(action),
        "blocked": bool(decision.blocked),
        "applied_action": int(decision.applied_action),
    })
assert all(item["blocked"] and item["applied_action"] == 0 for item in safety_checks)

# Memory is checked by reloading it in a fresh manager, not by trusting the training object.
exam_memory = MemoryManager()
exam_memory.load(str(MEMORY_PATH))
memory_exam_pass = (
    len(exam_memory.archive.events) == memory_roundtrip["events_after"]
    and len(exam_memory.skills.skills) == memory_roundtrip["skills_after"]
)

FINAL_EXAM_SECONDS = time.monotonic() - exam_start
assert sha256_file(CHECKPOINT_PATH) == FROZEN_CHECKPOINT_SHA256, "Checkpoint changed during final exam"

FINAL_EXAM = {
    "status": "COMPLETED",
    "exam_kind": "INDEPENDENT_EXPLORATORY_DEV_CHALLENGE",
    "world_model": FINAL_WM_REPORT["equal_episode_mean"],
    "brain_interfaces": brain_interface_exam,
    "memory_roundtrip_pass": memory_exam_pass,
    "safety_checks": safety_checks,
    "compressed_skill": COMPRESSED_REPORT["tasks"],
    "mapped_skill": MAPPED_REPORT["tasks"],
    "checkpoint_unchanged": True,
    "confirm_accessed": False,
    "wall_seconds": FINAL_EXAM_SECONDS,
}
assert memory_exam_pass
print("Independent final exam completed. See per-subsystem results below.")

In [ ]:
# 12) Produce the independent report, preserve negative results, and export to Drive.
horizons = FINAL_WM_REPORT["equal_episode_mean"]["horizons"]
wm_summary = {}
for h in ("1", "5", "20", "100"):
    row = horizons[h]
    model = row["anchored_model_mse"]
    baseline = row["persistence_mse"]
    wm_summary[h] = {
        "anchored_model_mse": model,
        "open_loop_model_mse": row["open_loop_model_mse"],
        "persistence_mse": baseline,
        "relative_improvement_vs_persistence": (
            None if model is None or baseline in (None, 0)
            else float((baseline - model) / baseline)
        ),
    }

REPORT = {
    "schema_version": 1,
    "run_id": RUN_ID,
    "repo_commit": ACTUAL_COMMIT,
    "runtime": RUNTIME,
    "time_budget_seconds": TIME_BUDGET_SECONDS,
    "elapsed_seconds": time.monotonic() - RUN_STARTED,
    "time_budget_exceeded": (time.monotonic() - RUN_STARTED) > TIME_BUDGET_SECONDS,
    "training": {
        "transitions": transition_count,
        "updates": TRAIN_REPORT["updates"],
        "wall_seconds": TRAIN_SECONDS,
        "final_losses": TRAIN_REPORT["final_losses"],
        "checkpoint_sha256": FROZEN_CHECKPOINT_SHA256,
        "validation_dev": {
            h: {
                "model_mse": validation_h[h]["anchored_model_mse"],
                "persistence_mse": validation_h[h]["persistence_mse"],
            } for h in ("1", "5", "20", "100")
        },
    },
    "final_exam": {
        "kind": "independent exploratory DEV exam; not CONFIRM",
        "world_model": wm_summary,
        "brain_interfaces": brain_interface_exam,
        "memory": memory_roundtrip,
        "safety": {
            "passed": all(x["blocked"] and x["applied_action"] == 0 for x in safety_checks),
            "checks": safety_checks,
        },
        "compressed_skill": COMPRESSED_REPORT["tasks"],
        "mapped_skill": MAPPED_REPORT["tasks"],
        "checkpoint_unchanged": sha256_file(CHECKPOINT_PATH) == FROZEN_CHECKPOINT_SHA256,
    },
    "not_assessed_or_not_implemented": [
        "language-model or open-ended text reasoning",
        "pixel-based perception",
        "validated OOD detection or calibrated uncertainty",
        "formal held-out CONFIRM generalization",
        "formal adaptive-compute or compute-savings claims",
        "promotion of Phase 1 Candidate skills",
    ],
    "integrity": {
        "protocol_validator_passed": True,
        "full_pytest_passed": True,
        "confirm_accessed": False,
        "source_revision_pinned": True,
    },
}
REPORT_PATH = RUN_DIR / "SOUL_FINAL_REPORT.json"
REPORT_PATH.write_text(json.dumps(REPORT, indent=2, ensure_ascii=False, allow_nan=False), encoding="utf-8")

# Copy the complete run directory only after all artifacts have been finalized.
shutil.copytree(RUN_DIR, DRIVE_RUN_DIR, dirs_exist_ok=True)
archive_path = shutil.make_archive(str(DRIVE_RUN_DIR), "zip", root_dir=RUN_DIR)
print(json.dumps({
    "report": str(DRIVE_RUN_DIR / REPORT_PATH.name),
    "archive": archive_path,
    "elapsed_minutes": round(REPORT["elapsed_seconds"] / 60, 1),
    "world_model": wm_summary,
    "memory_roundtrip": memory_roundtrip,
    "brain_interfaces": brain_interface_exam,
    "drive_folder": str(DRIVE_RUN_DIR),
}, indent=2, ensure_ascii=False))

## Completion checklist

- [ ] Protocol validator passed; full pytest suite passed.
- [ ] MiniCraft transitions collected from TRAIN only.
- [ ] Existing RSSM model trained and checkpoint hashed.
- [ ] Existing MemoryManager populated and round-trip verified.
- [ ] Compressed and mapped Candidate skill pipelines executed.
- [ ] Checkpoint frozen before final exam.
- [ ] Final exam used a separate DEV seed slice and never touched CONFIRM.
- [ ] Final JSON report and ZIP copied to Google Drive.

**Interpretation rule:** a lower world-model MSE is better; compare it against persistence at every horizon. A negative improvement is a real negative result. Skill success is reported separately from learned-agreement and fallback behavior. Passing interface or safety checks proves software behavior, not general intelligence.